In [17]:
import polars as pl
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import concurrent.futures
from tqdm import tqdm
import time
import os
import math
from afinn import Afinn
from collections import defaultdict
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/javclamar/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
def calcular_vader(texts):
    sia = SentimentIntensityAnalyzer()
    return [sia.polarity_scores(str(t))['compound'] for t in texts]

def calcular_afinn(texts):
    afinn = Afinn()
    scores = []
    alpha = 15
    for t in texts:
        raw = afinn.score(str(t))
        norm = raw / math.sqrt((raw**2) + alpha) if raw != 0 else 0.0
        scores.append(norm)
    return scores

def cargar_nrc_dict():
    nrc_map = defaultdict(set)
    try:
        with open("../data/lexicons/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt", "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3 and int(parts[2]) == 1:
                    nrc_map[parts[0]].add(parts[1])
    except Exception:
        pass
    return nrc_map

NRC_DICT = cargar_nrc_dict() 

def calcular_nrc(texts):
    emotions = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 
                'sadness', 'surprise', 'trust', 'positive', 'negative']
    
    results = {f'nrc_{e}': [] for e in emotions}
    
    if not NRC_DICT:
        empty = [0.0] * len(texts)
        for k in results: results[k] = empty
        return results

    for text in texts:
        words = word_tokenize(str(text).lower())
        total = len(words)
        counts = {e: 0.0 for e in emotions}
        
        if total > 0:
            for w in words:
                if w in NRC_DICT:
                    for e in NRC_DICT[w]:
                        counts[e] += 1
            for e in emotions:
                counts[e] /= total
        
        for e in emotions:
            results[f'nrc_{e}'].append(counts[e])
            
    return results

LEXICONS_AVAILABLE = {
    "vader_score": calcular_vader,
    "afinn_score": calcular_afinn,
    "nrc_emotions": calcular_nrc
}


def process_chunk(args):
    texts, active_models = args
    results = {}
    for model_name in active_models:
        if model_name in LEXICONS_AVAILABLE:
            results[model_name] = LEXICONS_AVAILABLE[model_name](texts)
    return results

In [23]:
csv_reviews = '../data/csv/yelp_academic_dataset_review.csv'
csv_reviews_output_scores = '../results/vader/yelp_academic_dataset_review_scored.csv'
batch_size = 200000
total_rows = 6_990_280

def analyze_sentiment(input_csv, output_csv, lexicons=['vader_score', 'afinn_score']):
    
    print(f"Lexicons a usar: {lexicons}")
    

    dummy_output = process_chunk((["test"], lexicons))
    
    new_column_names = []
    for lexicon in lexicons:
        res = dummy_output[lexicon]
        if isinstance(res, dict):
            new_column_names.extend(sorted(res.keys()))
        else:
            new_column_names.append(lexicon)
            
    try:
        temp_df = pl.read_csv(input_csv, n_rows=1, ignore_errors=True)
        final_columns = temp_df.columns + new_column_names
        
        with open(output_csv, 'w') as f:
            f.write(",".join([f'"{c}"' for c in final_columns]) + "\n")
    except Exception as e:
        print(f"Error headers: {e}")
        return

    num_cores = os.cpu_count()
    
    reader = pl.read_csv_batched(input_csv, batch_size=batch_size, ignore_errors=True)
    start_time = time.time()
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
        with tqdm(total=total_rows, unit="reviews", desc="Procesando") as pbar:
            while True:
                batches = reader.next_batches(1)
                if not batches: break
                
                df_batch = batches[0]
                texts = df_batch["text"].to_list()
                
                chunk_size = math.ceil(len(texts) / num_cores)
                chunks = [texts[i:i + chunk_size] for i in range(0, len(texts), chunk_size)]
                worker_args = [(chunk, lexicons) for chunk in chunks]
                
                results_generator = executor.map(process_chunk, worker_args)
                
                batch_data_flat = {col: [] for col in new_column_names}
                
                for res_dict in results_generator:
                    for lexicon in lexicons:
                        output = res_dict[lexicon]
                        
                        if isinstance(output, dict):
                            for sub_col in output:
                                batch_data_flat[sub_col].extend(output[sub_col])
                        else:
                            batch_data_flat[model].extend(output)
                
                df_scored = df_batch
                for col in new_column_names:
                    df_scored = df_scored.with_columns(
                        pl.Series(name=col, values=batch_data_flat[col], dtype=pl.Float64)
                    )
                
                df_scored.select(final_columns).write_csv(
                    file=open(output_csv, "a"),
                    include_header=False,
                    quote_style="always"
                )
                
                pbar.update(len(texts))

    print(f"Tiempo total: {(time.time() - start_time) / 60:.2f} min")

analyze_sentiment(csv_reviews, csv_reviews_output_scores, lexicons=['nrc_emotions'])

Lexicons a usar: ['nrc_emotions']


Procesando: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6990280/6990280 [07:49<00:00, 14881.33reviews/s]

Tiempo total: 7.83 min


In [24]:
csv_reviews_output_scores = '../results/vader/yelp_academic_dataset_review_scored.csv'

print(pl.scan_csv(csv_reviews_output_scores, ignore_errors=True).head(5).collect())

shape: (5, 17)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ review_id  ┆ user_id    ┆ business_i ┆ stars ┆ … ┆ nrc_posit ┆ nrc_sadne ┆ nrc_surpr ┆ nrc_trust │
│ ---        ┆ ---        ┆ d          ┆ ---   ┆   ┆ ive       ┆ ss        ┆ ise       ┆ ---       │
│ str        ┆ str        ┆ ---        ┆ i64   ┆   ┆ ---       ┆ ---       ┆ ---       ┆ f64       │
│            ┆            ┆ str        ┆       ┆   ┆ f64       ┆ f64       ┆ f64       ┆           │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ KU_O5udG6z ┆ mh_-eMZ6K5 ┆ XQfwVwDr-v ┆ 3     ┆ … ┆ 0.035088  ┆ 0.008772  ┆ 0.017544  ┆ 0.026316  │
│ pxOg-VcAEo ┆ RLWhZyISBh ┆ 0ZS3_CbbE5 ┆       ┆   ┆           ┆           ┆           ┆           │
│ dg         ┆ wA         ┆ Xw         ┆       ┆   ┆           ┆           ┆           ┆           │
│ BiTunyQ73a ┆ OyoGAe7OKp ┆ 7ATYjTIgM3 ┆ 5     ┆ … ┆ 0.057471  ┆ 0.0       ┆